# RAG 第 16 课：HyDE（Hypothetical Document Embeddings）

上一课我们让 LLM 改写 query 或生成多个 query 去检索。
这一课换一个思路，非常巧妙：

**先让 LLM 假装回答，再用那个"假答案"去检索。**

为什么这样有用？
- query 往往是短问句，和文档句式差得很远
- 文档是陈述句，嵌入空间里更接近"答案样子的句子"
- 所以用"假设答案"作为查询，embedding 召回往往更准

HyDE = Hypothetical Document Embeddings，思路就这么直接。

注意：HyDE 主要作用于 **dense 检索**。BM25 本身关心字面词，而假答案里常含文档里的关键词，所以 lexical 那一路也会间接受益。

In [ ]:
import gc
try:
    import torch
    if torch.cuda.is_available():
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available ({e})')

## 1. 连接 LM Studio（Qwen 3.6）

In [ ]:
import math, re
from collections import Counter, defaultdict
from typing import List

import numpy as np
from datasets import load_dataset
from openai import OpenAI

client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
print('available models:', model_ids)

preferred = ['qwen/qwen3.6-35b-a3b', 'qwen3.6-35b-a3b', 'qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower()), None)
print('chat =', chat_model, '| embed =', embedding_model)
assert chat_model and embedding_model

## 2. 数据 + BM25 + dense + RRF（沿用 14/15 课）

In [ ]:
raw_ds = load_dataset('squad', split='validation[:160]')
context_to_doc_id = {}
documents, eval_rows = [], []
for item in raw_ds:
    context = item['context'].strip()
    if context not in context_to_doc_id:
        context_to_doc_id[context] = len(documents)
        documents.append({'doc_id': len(documents), 'text': context, 'title': item['title']})
    doc_id = context_to_doc_id[context]
    if item['answers']['text']:
        eval_rows.append({
            'question': item['question'].strip(),
            'gold_doc_id': doc_id,
            'reference_answer': item['answers']['text'][0].strip(),
            'title': item['title'],
        })
print('documents =', len(documents), '| eval_rows =', len(eval_rows))

In [ ]:
def tokenize(text): return re.findall(r'[a-zA-Z0-9]+', text.lower())

class BM25:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.doc_lens = np.array([len(t) for t in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_lens.mean())
        self.N = len(corpus_tokens)
        self.postings = defaultdict(dict)
        for did, toks in enumerate(corpus_tokens):
            for tok, f in Counter(toks).items():
                self.postings[tok][did] = f
        self.idf = {tok: math.log(1 + (self.N - len(p) + 0.5)/(len(p) + 0.5)) for tok, p in self.postings.items()}
    def score(self, q_tokens):
        s = np.zeros(self.N, dtype=np.float32)
        for tok in q_tokens:
            if tok not in self.postings: continue
            idf = self.idf[tok]
            for did, f in self.postings[tok].items():
                dl = self.doc_lens[did]
                s[did] += idf * (f*(self.k1+1)) / (f + self.k1*(1 - self.b + self.b*dl/self.avgdl))
        return s

doc_tokens = [tokenize(d['text']) for d in documents]
bm25 = BM25(doc_tokens)

def get_embeddings(texts, batch_size=16):
    out = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(model=embedding_model, input=texts[i:i+batch_size])
        out.extend([np.array(x.embedding, dtype=np.float32) for x in resp.data])
    return np.vstack(out)

def l2n(m):
    n = np.linalg.norm(m, axis=1, keepdims=True); n = np.clip(n, 1e-12, None); return m/n

doc_embeddings = l2n(get_embeddings([d['text'] for d in documents]))
print('doc_embeddings =', doc_embeddings.shape)

In [ ]:
def lexical_retrieve(q, top_k=10):
    s = bm25.score(tokenize(q))
    idx = np.argsort(s)[::-1][:top_k]
    return [(int(i), float(s[i])) for i in idx]

def dense_retrieve_from_text(text, top_k=10):
    q_emb = l2n(get_embeddings([text]))[0]
    s = doc_embeddings @ q_emb
    idx = np.argsort(s)[::-1][:top_k]
    return [(int(i), float(s[i])) for i in idx]

def rrf_fuse(rankings_list, k=60, top_k=10):
    fused = defaultdict(float)
    for r in rankings_list:
        for rk, (did, _) in enumerate(r, start=1):
            fused[did] += 1.0/(k+rk)
    s = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    return [(d, sc) for d, sc in s[:top_k]]

def hybrid_retrieve(q, top_k=10, candidate_k=20):
    return rrf_fuse([lexical_retrieve(q, candidate_k), dense_retrieve_from_text(q, candidate_k)], top_k=top_k)

## 3. HyDE：让 LLM 先写一个"假答案"

关键点：
- 我们要的是一段**像答案的陈述句**，不是一个新的 query
- 长度控制在 1-3 句，过长会引入噪声
- 必须告诉 LLM**允许猜测**，不要让它说"I don't know"
- 温度可以略高一点，让假答案有点发散

In [ ]:
def generate_hypothetical_doc(question: str) -> str:
    system_prompt = (
        'You write a short hypothetical passage that would answer the question. '
        'The passage must read like a factual encyclopedia paragraph. '
        'Rules: (1) 1 to 3 sentences; '
        '(2) do not refuse; guess confidently even if you are unsure; '
        '(3) include likely named entities, dates and keywords; '
        '(4) do not mention that it is hypothetical; '
        '(5) output the passage only.'
    )
    resp = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f'Question: {question}\n\nHypothetical passage:'},
        ],
        temperature=0.5,
    )
    return resp.choices[0].message.content.strip()

demo_q = eval_rows[0]['question']
print('question :', demo_q)
print('hyde doc :', generate_hypothetical_doc(demo_q))

## 4. 三种 HyDE 用法

HyDE 到底拿假答案干什么？有三种常见做法：

1. **dense-only HyDE**：只用假答案的 embedding 检索
2. **hybrid HyDE**：把假答案同时喂给 BM25 + dense，再 RRF
3. **HyDE + 原 query 融合**：原 query 和假答案各走一遍 hybrid，再 RRF

第 3 种最稳，因为即使 HyDE 写歪了，原 query 还在兜底。

In [ ]:
def retrieve_baseline(q, top_k=3):
    return [d for d,_ in hybrid_retrieve(q, top_k=top_k)]

def retrieve_hyde_dense(q, top_k=3):
    hyp = generate_hypothetical_doc(q)
    return [d for d,_ in dense_retrieve_from_text(hyp, top_k=top_k)], hyp

def retrieve_hyde_hybrid(q, top_k=3, candidate_k=20):
    hyp = generate_hypothetical_doc(q)
    lex = lexical_retrieve(hyp, top_k=candidate_k)
    den = dense_retrieve_from_text(hyp, top_k=candidate_k)
    return [d for d,_ in rrf_fuse([lex, den], top_k=top_k)], hyp

def retrieve_hyde_plus_query(q, top_k=3, candidate_k=20):
    hyp = generate_hypothetical_doc(q)
    rankings = [
        lexical_retrieve(q, candidate_k),
        dense_retrieve_from_text(q, candidate_k),
        lexical_retrieve(hyp, candidate_k),
        dense_retrieve_from_text(hyp, candidate_k),
    ]
    return [d for d,_ in rrf_fuse(rankings, top_k=top_k)], hyp

## 5. 生成答案 + 评估函数

In [ ]:
def answer_with_llm(question, doc_ids):
    ctx = '\n\n'.join([f'[Document {i}] title: {documents[d]["title"]}\n{documents[d]["text"]}' for i,d in enumerate(doc_ids, start=1)])
    resp = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': 'Answer only from the context. If not supported, reply NOT_FOUND. Keep short.'},
            {'role': 'user', 'content': f'Question: {question}\n\nContext:\n{ctx}\n\nReturn only the answer.'},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

def norm_text(t):
    return ' '.join([re.sub(r'[^a-z0-9]+','', x.lower()) for x in t.split() if re.sub(r'[^a-z0-9]+','', x.lower())])

def em(p, r): return 1.0 if norm_text(p)==norm_text(r) else 0.0

def f1(p, r):
    pt = [t for t in norm_text(p).split() if t]
    rt = [t for t in norm_text(r).split() if t]
    if not pt or not rt: return 0.0
    ov = sum((Counter(pt)&Counter(rt)).values())
    if ov==0: return 0.0
    pr, rc = ov/len(pt), ov/len(rt)
    return 2*pr*rc/(pr+rc)

## 6. 单条样本先看一眼

In [ ]:
sample = eval_rows[0]
base = retrieve_baseline(sample['question'])
hd, hyp1 = retrieve_hyde_dense(sample['question'])
hh, hyp2 = retrieve_hyde_hybrid(sample['question'])
hq, hyp3 = retrieve_hyde_plus_query(sample['question'])

print('question   :', sample['question'])
print('gold_doc_id:', sample['gold_doc_id'])
print('hyde doc   :', hyp1)
print()
print('baseline      top-3:', base)
print('hyde-dense    top-3:', hd)
print('hyde-hybrid   top-3:', hh)
print('hyde+query    top-3:', hq)

## 7. 小规模评估：4 种策略

In [ ]:
eval_subset = eval_rows[:6]
results = []
for i, row in enumerate(eval_subset, start=1):
    print(f'processing {i}/{len(eval_subset)}: {row["question"]}')
    base = retrieve_baseline(row['question'])
    try: hd, _ = retrieve_hyde_dense(row['question'])
    except Exception: hd = base
    try: hh, _ = retrieve_hyde_hybrid(row['question'])
    except Exception: hh = base
    try: hq, _ = retrieve_hyde_plus_query(row['question'])
    except Exception: hq = base

    a_b = answer_with_llm(row['question'], base)
    a_hd = answer_with_llm(row['question'], hd)
    a_hh = answer_with_llm(row['question'], hh)
    a_hq = answer_with_llm(row['question'], hq)

    results.append({
        'q': row['question'], 'gold': row['gold_doc_id'], 'ref': row['reference_answer'],
        'base': base, 'hd': hd, 'hh': hh, 'hq': hq,
        'hit1': (base[0]==row['gold_doc_id'], hd[0]==row['gold_doc_id'], hh[0]==row['gold_doc_id'], hq[0]==row['gold_doc_id']),
        'hit3': (row['gold_doc_id'] in base, row['gold_doc_id'] in hd, row['gold_doc_id'] in hh, row['gold_doc_id'] in hq),
        'ans': (a_b, a_hd, a_hh, a_hq),
        'em': (em(a_b,row['reference_answer']), em(a_hd,row['reference_answer']), em(a_hh,row['reference_answer']), em(a_hq,row['reference_answer'])),
        'f1': (f1(a_b,row['reference_answer']), f1(a_hd,row['reference_answer']), f1(a_hh,row['reference_answer']), f1(a_hq,row['reference_answer'])),
    })

def mean_tup(key, idx): return float(np.mean([(1.0 if r[key][idx] else 0.0) if isinstance(r[key][idx], bool) else r[key][idx] for r in results]))

labels = ['baseline', 'hyde-dense', 'hyde-hybrid', 'hyde+query']
print('\n=== summary ===')
for metric in ['hit1','hit3','em','f1']:
    print(metric, {labels[i]: round(mean_tup(metric, i), 3) for i in range(4)})

## 8. 工程直觉

1. **HyDE 最怕的是"一本正经地胡说"**：假答案里如果混入完全不相关的实体，embedding 会被带偏。
2. **HyDE + 原 query 融合几乎不会掉**：因为原 query 兜底。
3. 适合 HyDE 的场景：用户 query 非常短，或者 query 和文档风格差距很大（比如用户口语 vs 技术文档）。
4. 不适合 HyDE 的场景：query 已经很长、信息很完整，HyDE 反而会增加噪声。

下一课：**Contextual Compression**，让 LLM 在生成答案前先把检索到的大段文字"压缩到只剩相关的部分"，省 token 又更准。